# Candès, Li, Ma & Wright (2011) — Robust PCA via Principal Component Pursuit / RPCA 구현

**Paper**: Candès, E. J., Li, X., Ma, Y., & Wright, J. (2011). *Robust Principal Component Analysis?* Journal of the ACM, **58**(3), Article 11.

This notebook implements the paper's main algorithm — **Principal Component Pursuit by Augmented Lagrange Multiplier (ALM/ADMM)** — and reproduces a small version of two flagship experiments:

1. The **shrinkage operators** (soft-threshold $\mathcal S_\tau$ and singular-value-threshold $\mathcal D_\tau$).
2. The **PCP / ALM** solver (Algorithm 1 of Section 5).
3. **Random matrix demo**: exact recovery of a low-rank + sparse synthetic matrix.
4. **Phase-transition sweep** in $(r/n,\ \rho_s)$ — a small reproduction of Fig. 1.
5. **Video-style background subtraction** on a synthetic 5-frame sequence ($L_0 = $ static background, $S_0 = $ moving objects).
6. **Summary table** mapping the paper's concepts to modern equivalents.

이 노트북은 RPCA / PCP 의 ALM 해법을 처음부터 구현하고, 무작위 행렬 정확 회복·간단한 위상 전이·합성 비디오 배경 분리 세 실험으로 검증한다.

In [ ]:
from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## Part 1: Shrinkage operators / 임계처리 연산자

Two proximal operators power the ADMM iteration:

- **Soft thresholding**: $\mathcal S_\tau[x] = \mathrm{sgn}(x)\max(|x|-\tau,0)$, the proximal of $\tau\|\cdot\|_1$.
- **Singular-value thresholding**: $\mathcal D_\tau(X) = U\,\mathcal S_\tau(\Sigma)\,V^*$ where $X = U\Sigma V^*$, the proximal of $\tau\|\cdot\|_*$ (Cai-Candès-Shen 2010).

In [ ]:
def soft_threshold(x: np.ndarray, tau: float) -> np.ndarray:
    """Element-wise soft thresholding: sgn(x)*max(|x|-tau, 0).

    Args:
        x: Input array.
        tau: Non-negative threshold.

    Returns:
        Soft-thresholded array, same shape as x.
    """
    return np.sign(x) * np.maximum(np.abs(x) - tau, 0.0)


def singular_value_threshold(X: np.ndarray, tau: float) -> Tuple[np.ndarray, int]:
    """Singular value thresholding D_tau(X) = U * S_tau(sigma) * V^T.

    Args:
        X: 2-D matrix.
        tau: Non-negative threshold on singular values.

    Returns:
        L: Thresholded low-rank matrix (same shape as X).
        rank: Effective rank (number of singular values exceeding tau).
    """
    U, sigma, Vt = np.linalg.svd(X, full_matrices=False)
    sigma_t = soft_threshold(sigma, tau)
    rank = int((sigma_t > 0).sum())
    return (U[:, :rank] * sigma_t[:rank]) @ Vt[:rank, :], rank


# Sanity check — singular values shrunk / 특이값이 줄어드는지 확인
test = rng.standard_normal((6, 6))
L_test, rk = singular_value_threshold(test, 0.5)
sigma_orig = np.linalg.svd(test, compute_uv=False)
sigma_new = np.linalg.svd(L_test, compute_uv=False)
print('original sigmas:', np.round(sigma_orig, 3))
print('thresholded sigmas:', np.round(sigma_new, 3))
print('rank:', rk)

## Part 2: Principal Component Pursuit by ALM / ALM 으로 PCP 풀기

Algorithm 1 of Section 5 — alternating direction:

$$
\begin{aligned}
L_{k+1} &= \mathcal D_{\mu^{-1}}\!\left(M - S_k - \mu^{-1}Y_k\right),\\
S_{k+1} &= \mathcal S_{\lambda\mu^{-1}}\!\left(M - L_{k+1} + \mu^{-1}Y_k\right),\\
Y_{k+1} &= Y_k + \mu\big(M - L_{k+1} - S_{k+1}\big).
\end{aligned}
$$

Stopping: $\|M - L - S\|_F \le \delta\|M\|_F$ with $\delta = 10^{-7}$. Recommended $\mu = n_1 n_2 / (4\|M\|_1)$.

In [ ]:
@dataclass
class PCPResult:
    """Container for PCP / RPCA output."""
    L: np.ndarray
    S: np.ndarray
    Y: np.ndarray
    n_iter: int
    converged: bool
    history: list[float]


def principal_component_pursuit(
    M: np.ndarray,
    lam: float | None = None,
    mu: float | None = None,
    tol: float = 1e-7,
    max_iter: int = 500,
    verbose: bool = False,
) -> PCPResult:
    """Solve min ||L||_* + lambda ||S||_1 s.t. L + S = M via ALM/ADMM.

    Implements Algorithm 1 of Section 5 (Candès et al. 2011).

    Args:
        M: Observed matrix.
        lam: Sparsity weight; default is 1/sqrt(max(n1, n2)).
        mu: Augmented Lagrangian penalty; default n1*n2 / (4 * ||M||_1).
        tol: Stopping tolerance ||M - L - S||_F / ||M||_F.
        max_iter: Maximum ADMM iterations.
        verbose: Print residual every 10 iterations.

    Returns:
        PCPResult with L, S, Y, iteration count and history.
    """
    n1, n2 = M.shape
    if lam is None:
        lam = 1.0 / np.sqrt(max(n1, n2))
    M_l1 = float(np.sum(np.abs(M)))
    if mu is None:
        mu = n1 * n2 / max(4.0 * M_l1, 1e-12)
    M_norm = float(np.linalg.norm(M, 'fro'))

    L = np.zeros_like(M)
    S = np.zeros_like(M)
    Y = np.zeros_like(M)
    history: list[float] = []

    for k in range(1, max_iter + 1):
        L, _ = singular_value_threshold(M - S - Y / mu, 1.0 / mu)
        S = soft_threshold(M - L + Y / mu, lam / mu)
        residual = M - L - S
        Y = Y + mu * residual
        rel_res = float(np.linalg.norm(residual, 'fro')) / max(M_norm, 1e-12)
        history.append(rel_res)
        if verbose and (k % 10 == 0 or k == 1):
            print(f'iter {k:4d}: rel residual = {rel_res:.3e}')
        if rel_res < tol:
            return PCPResult(L=L, S=S, Y=Y, n_iter=k, converged=True, history=history)
    return PCPResult(L=L, S=S, Y=Y, n_iter=max_iter, converged=False, history=history)

## Part 3: Random matrix demo / 무작위 행렬 정확 회복

Generate a square matrix $M = L_0 + S_0$ with $L_0 = X Y^*$, $X, Y \in \mathbb R^{n \times r}$ i.i.d. $\mathcal N(0, 1/n)$, and $S_0$ Bernoulli $\pm 1$ on a random support of size $\rho_s n^2$. Run PCP and confirm exact recovery (relative error $< 10^{-4}$) — a small reproduction of Table 1 of the paper.

In [ ]:
def make_lowrank_sparse(n: int, r: int, rho_s: float, seed: int = 0) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate a low-rank + sparse synthetic test matrix.

    Args:
        n: Matrix dimension (square).
        r: Rank of L0.
        rho_s: Fraction of entries in support of S0.
        seed: Reproducible seed.

    Returns:
        L0: Rank-r matrix with i.i.d. N(0, 1/n) factors.
        S0: Sparse matrix with +/- 1 entries on a uniform random support.
        M: L0 + S0.
    """
    g = np.random.default_rng(seed)
    X = g.standard_normal((n, r)) / np.sqrt(n)
    Y = g.standard_normal((n, r)) / np.sqrt(n)
    L0 = X @ Y.T
    mask = g.random((n, n)) < rho_s
    S0 = np.where(mask, g.choice([-1.0, 1.0], size=(n, n)), 0.0)
    return L0, S0, L0 + S0


n = 100
r = 5
rho_s = 0.10
L0, S0, M = make_lowrank_sparse(n=n, r=r, rho_s=rho_s, seed=2026)

t0 = time.perf_counter()
result = principal_component_pursuit(M, tol=1e-7, max_iter=500, verbose=False)
elapsed = time.perf_counter() - t0

rel_L = np.linalg.norm(result.L - L0, 'fro') / np.linalg.norm(L0, 'fro')
rel_S = np.linalg.norm(result.S - S0, 'fro') / max(np.linalg.norm(S0, 'fro'), 1e-12)
rank_recovered = int((np.linalg.svd(result.L, compute_uv=False) > 1e-3).sum())
support_recovered = int((np.abs(result.S) > 1e-3).sum())
print(f'Dimension n = {n}, rank(L0) = {r}, ||S0||_0 = {int((S0 != 0).sum())}')
print(f'Recovered rank(L_hat)  = {rank_recovered}')
print(f'Recovered ||S_hat||_0 = {support_recovered}')
print(f'||L_hat - L0||_F / ||L0||_F = {rel_L:.3e}')
print(f'||S_hat - S0||_F / ||S0||_F = {rel_S:.3e}')
print(f'Iterations = {result.n_iter}, converged = {result.converged}, elapsed = {elapsed:.2f} s')

plt.figure(figsize=(7, 3.5))
plt.semilogy(result.history)
plt.xlabel('iteration'); plt.ylabel('||M - L - S||_F / ||M||_F')
plt.title('PCP convergence (random matrix demo)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Part 4: Phase transition (small) / 위상 전이 (소규모 재현)

Reproduce the qualitative shape of Fig. 1 of the paper. Sweep $r/n \in [0.05, 0.4]$ and $\rho_s \in [0.05, 0.4]$; declare success if $\|\hat L - L_0\|_F / \|L_0\|_F < 10^{-3}$. We use $n = 60$ for tractability and a single trial per cell.

In [ ]:
n_pt = 60
ranks = np.arange(2, 18, 3)             # r/n in [0.03, 0.28]
rhos = np.linspace(0.05, 0.4, 6)
phase = np.zeros((len(ranks), len(rhos)))
for i, r_ in enumerate(ranks):
    for j, rho_ in enumerate(rhos):
        L0_, S0_, M_ = make_lowrank_sparse(n=n_pt, r=int(r_), rho_s=float(rho_), seed=int(100 + 7 * i + j))
        res = principal_component_pursuit(M_, tol=1e-6, max_iter=300)
        rel = np.linalg.norm(res.L - L0_, 'fro') / max(np.linalg.norm(L0_, 'fro'), 1e-12)
        phase[i, j] = 1.0 if rel < 1e-3 else 0.0

plt.figure(figsize=(6, 4))
plt.imshow(phase, origin='lower', extent=[rhos[0], rhos[-1], ranks[0]/n_pt, ranks[-1]/n_pt], aspect='auto', cmap='gray', vmin=0, vmax=1)
plt.colorbar(label='success (1) / failure (0)')
plt.xlabel('sparsity ρ_s'); plt.ylabel('rank / n')
plt.title(f'Small phase-transition diagram (n = {n_pt})')
plt.tight_layout(); plt.show()

## Part 5: Synthetic background subtraction / 합성 비디오 배경 분리

Build a 5-frame, $48\times 48$ "video" with a static smooth background and a small bright moving square. Stack frames as columns of $M \in \mathbb R^{2304 \times 5}$. Apply PCP — $\hat L$ should recover the static background and $\hat S$ should isolate the moving square.

In [ ]:
def make_synthetic_video(n_frames: int = 5, h: int = 48, w: int = 48) -> Tuple[np.ndarray, np.ndarray]:
    """Make a tiny synthetic surveillance video.

    Args:
        n_frames: Number of frames in the sequence.
        h: Frame height in pixels.
        w: Frame width in pixels.

    Returns:
        frames: Array of shape (n_frames, h, w).
        bg: The static background frame, shape (h, w).
    """
    yy, xx = np.mgrid[0:h, 0:w]
    bg = 0.4 * np.exp(-((yy - h / 2) ** 2 + (xx - w / 2) ** 2) / (h * w / 6.0)) + 0.2
    frames = np.tile(bg, (n_frames, 1, 1))
    # moving bright square / 움직이는 사각형
    sq = 6
    for k in range(n_frames):
        cy = 8 + 6 * k
        cx = 8 + 6 * k
        frames[k, cy:cy + sq, cx:cx + sq] += 0.7
    return frames, bg


n_frames, h, w = 5, 48, 48
frames, bg = make_synthetic_video(n_frames=n_frames, h=h, w=w)
M_video = frames.reshape(n_frames, -1).T  # (h*w, n_frames)

res_v = principal_component_pursuit(M_video, tol=1e-6, max_iter=500)
L_frames = res_v.L.T.reshape(n_frames, h, w)
S_frames = res_v.S.T.reshape(n_frames, h, w)
print(f'Video PCP iterations = {res_v.n_iter}, converged = {res_v.converged}')
print(f'Recovered rank(L) = {int((np.linalg.svd(res_v.L, compute_uv=False) > 1e-3).sum())}')

fig, axes = plt.subplots(3, n_frames, figsize=(2.4 * n_frames, 7))
for k in range(n_frames):
    axes[0, k].imshow(frames[k], cmap='gray', vmin=0, vmax=1.2)
    axes[0, k].set_title(f'frame {k}'); axes[0, k].axis('off')
    axes[1, k].imshow(L_frames[k], cmap='gray', vmin=0, vmax=1.2)
    axes[1, k].set_title('L̂ (background)'); axes[1, k].axis('off')
    axes[2, k].imshow(np.abs(S_frames[k]), cmap='hot')
    axes[2, k].set_title('|Ŝ| (foreground)'); axes[2, k].axis('off')
plt.tight_layout(); plt.show()

## Part 6: Sensitivity to λ / λ 에 대한 민감도

Sweep the sparsity weight $\lambda$ around the universal $\lambda^* = 1/\sqrt{n_{(1)}}$ and observe how recovery accuracy changes. The paper claims the universal choice is good across a *wide* range of problems — verify on our random matrix.

In [ ]:
n_s = 80
L0_s, S0_s, M_s = make_lowrank_sparse(n=n_s, r=4, rho_s=0.10, seed=2025)
lam_star = 1.0 / np.sqrt(n_s)
lam_grid = lam_star * np.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0])
errs_L = []
errs_S = []
for lam in lam_grid:
    r_ = principal_component_pursuit(M_s, lam=float(lam), tol=1e-6, max_iter=300)
    errs_L.append(np.linalg.norm(r_.L - L0_s, 'fro') / np.linalg.norm(L0_s, 'fro'))
    errs_S.append(np.linalg.norm(r_.S - S0_s, 'fro') / np.linalg.norm(S0_s, 'fro'))

plt.figure(figsize=(7, 4))
plt.semilogy(lam_grid / lam_star, errs_L, 'o-', label='||L̂−L₀||/||L₀||')
plt.semilogy(lam_grid / lam_star, errs_S, 's-', label='||Ŝ−S₀||/||S₀||')
plt.axvline(1.0, color='k', ls=':')
plt.xlabel('λ / λ*'); plt.ylabel('relative Frobenius error')
plt.title('Sensitivity to λ (λ* = 1/√n is universal)')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()
for lam, eL, eS in zip(lam_grid, errs_L, errs_S):
    print(f'λ/λ* = {lam/lam_star:.2f}: errL = {eL:.2e}, errS = {eS:.2e}')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Convex relaxation of rank / 랭크 볼록 완화 | Nuclear norm $\|L\|_*$ | Trace norm regularisation in deep matrix factorisation |
| Convex relaxation of sparsity / 희소성 볼록 완화 | $\ell_1$ norm $\|S\|_1$ | $\ell_1$ regularisation in compressed sensing / Lasso |
| Universal regularisation / 보편 정규화 | $\lambda = 1/\sqrt{n_{(1)}}$ tuning-free | Tuning-free / self-tuning regularisation in DL |
| Inner solver / 내부 해법 | ADMM with closed-form proximals | Operator splitting; Plug-and-Play priors; LISTA |
| Background-subtraction example / 배경 분리 | Airport / lobby video | Modern surveillance & dynamic-MRI low-rank+sparse models |
| Solar physics use / 태양물리 응용 | (Not in paper) | K-corona vs CME separation in LASCO/SECCHI/Metis (Wang+ 2017; Lamy+ 2020) |
| Robust matrix completion / 결측+손상 회복 | Theorem 1.2 with $\lambda = 1/\sqrt{0.1\,n}$ | Robust collaborative filtering; tensor RPCA |